# Neural training and evaluation API

This notebook demonstrates the common run interface for Deep CFR, neural CFR+, neural FSP, asynchronous exact evaluation, and blocking approximate best-response evaluation. Long-running cells are disabled by default.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'liars_poker').is_dir():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from liars_poker.core import GameSpec
from liars_poker.eval.neural_evaluators import (
    AsyncExactExploitabilityEvaluator,
    BlockingFunctionEvaluator,
    ScheduledEvaluation,
)
from liars_poker.training.br_runs import (
    BestResponseConfig,
    BestResponseSuiteEvaluator,
    run_best_responder,
)
from liars_poker.training.neural_runs import (
    run_deep_cfr,
    run_neural_cfr_plus,
    run_neural_fsp,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

In [ ]:
spec = GameSpec(
    ranks=4,
    suits=4,
    hand_size=2,
    claim_kinds=('RankHigh', 'Pair', 'TwoPair', 'Trips'),
    suit_symmetry=True,
)

RUN_MINUTES = 45
RUN_DEEP_CFR = False
RUN_CFR_PLUS = False
RUN_FSP_EXAMPLE = False
RUN_BR_EXAMPLE = False

deep_cfr_kwargs = {
    'advantage_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': 7,
    'advantage_buffer_capacity': 2_000_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'advantage_train_steps': 100,
    'strategy_train_steps': 50,
    'advantage_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'highest_regret_fallback': True,
    'alternating_updates': True,
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

cfr_plus_kwargs = {
    'regret_hidden_sizes': (2048, 2048),
    'strategy_hidden_sizes': (512, 512),
    'device': device,
    'seed': 7,
    'regret_buffer_capacity': 500_000,
    'strategy_buffer_capacity': 2_000_000,
    'learning_rate': 1e-3,
    'batch_size': 4096,
    'regret_train_steps': 100,
    'strategy_train_steps': 50,
    'regret_positive_weight': 0.5,
    'strategy_weighting': 'linear',
    'validation_fraction': 0.02,
    'validation_buffer_capacity': 20_000,
    'traversal_backend': 'gpu_native',
    'traversal_batch_size': 1024,
    'device_replay': True,
    'fused_optimizer': True,
    'amp_dtype': None,
    'compile_models': False,
}

Exact exploitability jobs are queued in CPU worker processes. The measured training budget excludes policy serialization, exact evaluation, checkpoints, and time spent waiting for the queue at the end.

In [ ]:
def exact_schedule(every_minutes=5, workers=1):
    return ScheduledEvaluation(
        name='exact_exploitability',
        evaluator=AsyncExactExploitabilityEvaluator(
            max_workers=workers,
            compile_batch_size=65_536,
        ),
        every_minutes=every_minutes,
        run_at_end=True,
    )

deep_cfr_result = None
if RUN_DEEP_CFR:
    deep_cfr_result = run_deep_cfr(
        spec,
        minutes=RUN_MINUTES,
        traversals_per_player=1024,
        trainer_kwargs=deep_cfr_kwargs,
        evaluations=[exact_schedule(every_minutes=5)],
        checkpoint_every_minutes=15,
        debug=True,
    )
    print(deep_cfr_result.run_dir)

In [ ]:
cfr_plus_result = None
if RUN_CFR_PLUS:
    cfr_plus_result = run_neural_cfr_plus(
        spec,
        minutes=RUN_MINUTES,
        traversals_per_player=1024,
        trainer_kwargs=cfr_plus_kwargs,
        evaluations=[exact_schedule(every_minutes=5)],
        checkpoint_every_minutes=15,
        debug=True,
    )
    print(cfr_plus_result.run_dir)

Approximate responders use the same schedule abstraction but run synchronously on the single GPU. Main training pauses until every configured responder and seed finishes, then resumes. The example below can be passed alongside the asynchronous exact schedule.

In [ ]:
approximate_br_schedule = ScheduledEvaluation(
    name='approximate_br_suite',
    evaluator=BlockingFunctionEvaluator(
        BestResponseSuiteEvaluator([
            BestResponseConfig(
                method='action_conditioned_fitted_return',
                seeds=(7, 17, 27),
                minutes=15,
                trainer_kwargs={
                    'state_hidden_sizes': (512, 512),
                    'action_hidden_sizes': (128, 128),
                    'embedding_dim': 256,
                    'device': device,
                    'replay_capacity': 1_000_000,
                    'batch_size': 4096,
                    'learning_rate': 1e-3,
                    'train_steps': 100,
                    'warmup_transitions': 20_000,
                    'epsilon_start': 1.0,
                    'epsilon_end': 0.05,
                    'epsilon_decay_decisions': 500_000,
                    'rollouts_per_action': 1,
                },
                episodes_per_role=4096,
                rollout_batch_size=1024,
                eval_episodes_per_role=200_000,
                exact_evaluation=False,
            ),
        ])
    ),
    every_minutes=30,
    run_at_end=True,
)

# Example:
# evaluations=[exact_schedule(5), approximate_br_schedule]

In [ ]:
fsp_result = None
if RUN_FSP_EXAMPLE:
    fsp_result = run_neural_fsp(
        spec,
        iterations=3,
        trainer_kwargs={
            'average_state_hidden_sizes': (512, 512),
            'average_action_hidden_sizes': (128, 128),
            'average_embedding_dim': 256,
            'strategy_buffer_capacity': 1_000_000,
            'validation_buffer_capacity': 20_000,
            'validation_fraction': 0.02,
            'strategy_batch_size': 4096,
            'strategy_train_steps': 100,
            'strategy_learning_rate': 1e-3,
            'device': device,
            'seed': 7,
            'br_kwargs': {
                'state_hidden_sizes': (512, 512),
                'action_hidden_sizes': (128, 128),
                'embedding_dim': 256,
                'replay_capacity': 1_000_000,
                'batch_size': 4096,
                'learning_rate': 1e-3,
                'train_steps': 100,
                'warmup_transitions': 20_000,
                'epsilon_start': 1.0,
                'epsilon_end': 0.05,
                'epsilon_decay_decisions': 250_000,
                'rollouts_per_action': 1,
            },
        },
        br_iterations=100,
        br_episodes_per_role=512,
        br_rollout_batch_size=256,
        strategy_episodes_per_role=512,
        strategy_collection_batch_size=256,
        strategy_train_steps=100,
        evaluations=[exact_schedule(every_minutes=5)],
        checkpoint_every_iterations=1,
        debug=True,
    )

In [ ]:
# Train one responder directly against any Policy or saved run directory.
br_result = None
if RUN_BR_EXAMPLE:
    target = cfr_plus_result.policy if cfr_plus_result is not None else (
        REPO_ROOT / 'artifacts' / 'deep_cfr_plus_reference_runs'
        / 'r4_s4_h2_hp2pt_ss___20260621-024038'
    )
    br_result = run_best_responder(
        target,
        method='action_conditioned_fitted_return',
        minutes=15,
        trainer_kwargs={
            'state_hidden_sizes': (512, 512),
            'action_hidden_sizes': (128, 128),
            'embedding_dim': 256,
            'device': device,
            'seed': 7,
            'replay_capacity': 1_000_000,
            'batch_size': 4096,
            'learning_rate': 1e-3,
            'train_steps': 100,
            'warmup_transitions': 20_000,
            'epsilon_start': 1.0,
            'epsilon_end': 0.05,
            'epsilon_decay_decisions': 500_000,
            'rollouts_per_action': 1,
        },
        episodes_per_role=4096,
        rollout_batch_size=1024,
        evaluate_every_minutes=1,
        eval_episodes_per_role=200_000,
        exact_evaluation=True,
        debug=True,
    )
    pd.DataFrame(br_result.evaluation_records)

In [ ]:
# Exact resume: the supplied budget is additional to the completed run.
# resumed = run_neural_cfr_plus(
#     spec,
#     minutes=30,
#     traversals_per_player=1024,
#     resume_from=cfr_plus_result.run_dir,
#     device=device,
#     evaluations=[exact_schedule(every_minutes=5)],
#     checkpoint_every_minutes=15,
#     debug=True,
# )

In [ ]:
def plot_exact_results(result):
    frame = pd.DataFrame(result.evaluation_records)
    frame = frame[frame['evaluator'] == 'exact_exploitability'].copy()
    if frame.empty:
        return frame
    frame['training_minutes'] = frame['measured_training_s'] / 60.0
    ax = frame.plot(
        x='training_minutes',
        y='exploitability',
        marker='o',
        logy=True,
        legend=False,
        title='Exact exploitability',
    )
    ax.set_xlabel('Measured neural-training minutes')
    ax.set_ylabel('Exploitability')
    ax.grid(True, which='both', alpha=0.3)
    plt.show()
    return frame

# plot_exact_results(cfr_plus_result)